# 04 · EICV7 and FinScope 2024: data inspection

A read-only inspection of all 21 EICV7 cross-section files and the FinScope Rwanda 2024 file. Its purpose is to understand the data before building anything.

| | |
|---|---|
| **Inputs** | `Microdata/Cross_Section/*.dta` (EICV7) and `FINSCOPE_DATA/FinScope 2024 Rwanda.dta`. Both are read only and never modified. |
| **Outputs** | `reports/data_inspection/*.csv` and the summary in `reports/eicv7_finscope_inventory.md` |
| **Code** | `src/data/survey_inspection.py` (profiling) and `src/data/eicv7_cpi_mapping.py` (expenditure-to-CPI mapping) |
| **Runtime** | About 1–2 minutes. The 1.1 GB food file is read in chunks. |

**Scope.** This notebook only describes the data. It does not clean, recode, impute, deduplicate or model anything. Where it finds a problem, it reports it and recommends a fix for the next stage.

**Privacy.** The outputs contain metadata and aggregate statistics only. No household or person records are written out, and string variables are summarised by length and number of distinct values, never by example values.

In [ ]:
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data" / "survey_inspection.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.data import eicv7_cpi_mapping as cpimap
from src.data import survey_inspection as si
from IPython.display import display

OUT = si.REPORT_DIR
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.float_format", "{:,.2f}".format)
print(f"{len(si.DATASETS)} datasets registered; outputs go to {OUT.relative_to(ROOT)}/")

## 1. Profile every dataset

Each file is read once, in chunks. Variable statistics come from the stored codes, not the value labels. Identifiers, geography, weights and a few check variables are kept in memory for the cross-file checks in section 5.

In [ ]:
COMMON = ["hhid", "pid", "clust", "strata_id", "province", "district", "ur", "weight", "pop_wt",
          "quintile", "poverty", "pov_jan", "epov_jan"]
EXTRA = {
    "poverty": ["member", "ae", "cons1", "cons1ae", "sol_jan", "hh_index", "food", "Food_at_school",
                *[f"exp{i}" for i in range(1, 12)], "Poverty_line", "Extreme_line"],
    "person": ["s1q1", "s1q2", "s1q3y"],
    "durables": ["s10bq0", "s10bq2", "s10bq5a"],
    "nonfood_annual": ["s8a1q0", "s8a1q2", "s8a1q3"],
    "nonfood_monthly": ["s8a2q0", "s8a2q2", "s8a2q3"],
    "nonfood_weekly": ["s8a3q0", "s8a3q2", "s8a3q3"],
    "food": ["s8bq0", *[f"s8bq4_v{v}" for v in range(2, 6)]],
    "transfers_out": ["s9aq0"],
    "transfers_in": ["s9bq0"],
    "other_expenditure": ["s9cq1", "s9cq2"],
    "vup_direct_support": ["s9d1q1", "s9d1q2", "s9d1q4"],
    "vup_classic_pw": ["s9d1q1", "s9d2q1", "s9d2q11"],
    "vup_expanded_pw": ["s9d1q1", "s9d3q1", "s9d3q10"],
    "vup_nsds": ["s9d1q1", "s9d4q1", "s9d4q7", "s9d4q8", "s9d4q8a"],
    "vup_financial_services": ["s9d1q1", "s9d5q1", "s9d5q2", "s9d5q4"],
    "other_income": ["s9eq1", "s9eq4"],
    "finscope": ["a1", "a2", "a6", "hh_wt", "b1"],
}

profiles = {}
for ds in si.DATASETS:
    start = time.time()
    keep = list(dict.fromkeys(COMMON + list(ds.key) + EXTRA.get(ds.name, [])))
    profiles[ds.name] = si.profile_dataset(ds, keep=keep)
    print(f"{ds.survey:14} {ds.name:24} {profiles[ds.name].n_rows:>10,} rows  {time.time() - start:5.1f}s")

## 2. Dataset inventory

In [ ]:
inventory = []
for name, p in profiles.items():
    ds, v, f = p.dataset, p.variables, p.frame
    keys = si.key_check(f, ds.key)
    unit_id = "hhid" if "hhid" in f else "key"
    per_hh = f.groupby(unit_id).size()
    inventory.append({
        "survey": ds.survey, "dataset": name, "file": str(ds.path.relative_to(ROOT)),
        "size_mb": round(ds.path.stat().st_size / 1e6, 1), "rows": p.n_rows, "variables": len(v),
        "unit_of_analysis": ds.unit, "key": keys["key"], "key_unique": keys["key_unique"],
        "duplicate_key_rows": keys["n_duplicate_key_rows"], "exact_duplicate_rows": p.n_exact_duplicate_rows,
        "households": f[unit_id].nunique(), "rows_per_household_median": per_hh.median(),
        "rows_per_household_max": per_hh.max(),
        "weights": ", ".join(v.loc[v["role"] == "weight", "variable"]),
        "geography_and_design": ", ".join(v.loc[v["role"].isin(["geography", "design"]), "variable"]),
        "pct_cells_missing": round(100 * v["n_missing"].sum() / (p.n_rows * len(v)), 1),
        "description": ds.description,
    })
inventory = pd.DataFrame(inventory)
inventory.to_csv(OUT / "dataset_inventory.csv", index=False)
print(f"{len(inventory)} files, {inventory['rows'].sum():,} rows, {inventory['variables'].sum():,} variables, "
      f"{inventory['size_mb'].sum():,.0f} MB")
inventory[["survey", "dataset", "size_mb", "rows", "variables", "unit_of_analysis", "key", "key_unique",
           "duplicate_key_rows", "exact_duplicate_rows", "households", "rows_per_household_max", "pct_cells_missing"]]

## 3. Draft data dictionary

One row per variable in every file: label, storage type, value labels, missingness and summary statistics. Each variable is tagged with an analysis topic, which rolls up into the eight focus areas of this inspection. The tags are assigned by rules on module and variable name, so treat them as a first draft to review.

In [ ]:
FOCUS_AREAS = {
    "identifier": "1. Identifiers", "survey_weight": "2. Survey weights", "geography": "3. Geography and design",
    "poverty_consumption": "4. Poverty and consumption", "food_expenditure": "5. Food expenditure",
    "nonfood_expenditure": "5. Non-food expenditure", "social_protection": "6. VUP and social protection",
    "financial_inclusion": "7. Financial inclusion", "financial_capability": "7. Financial inclusion",
    "income_transfers": "8. Income and transfers", "employment": "8. Employment", "education": "8. Education",
    "assets": "8. Assets", "housing": "8. Housing and utilities", "service_access": "8. Service access",
    "demographics": "Other: demographics", "health": "Other: health", "ict": "Other: ICT",
    "shocks": "Other: shocks", "agriculture": "Other: agriculture", "other": "Other",
}
COLUMNS = ["survey", "dataset", "position", "variable", "label", "focus_area", "topic", "role", "stata_type", "kind",
           "value_label_name", "n_value_labels", "value_labels", "n_rows", "n_missing", "pct_missing", "n_unique",
           "min", "max", "mean", "std", "n_zero", "n_negative", "n_dk_refused", "dk_refused_codes",
           "n_unlabelled_values", "unlabelled_codes", "n_possible_sentinel", "max_length"]

dictionary = pd.concat([p.variables for p in profiles.values()], ignore_index=True)
dictionary["focus_area"] = dictionary["topic"].map(FOCUS_AREAS)
assert dictionary["focus_area"].notna().all()
dictionary = dictionary.reindex(columns=COLUMNS)
dictionary.to_csv(OUT / "data_dictionary.csv", index=False)
print(f"{len(dictionary):,} variables documented")
dictionary.pivot_table(index="focus_area", columns="survey", values="variable", aggfunc="count", fill_value=0)

### Key variables by focus area

The table below picks out the variables that matter most for each focus area. The full list is in `data_dictionary.csv`.

In [ ]:
KEY_VARIABLES = {
    "1. Identifiers": [("poverty", "hhid"), ("person", "pid"), ("credit", "s10aq4"), ("savings", "s10cq3"), ("finscope", "key")],
    "2. Survey weights": [("poverty", "weight"), ("poverty", "pop_wt"), ("poverty", "clust"), ("durables", "strata_id"),
                          ("finscope", "hh_wt"), ("finscope", "pop_wt")],
    "3. Geography": [("poverty", "province"), ("poverty", "district"), ("poverty", "ur"),
                     ("finscope", "a1"), ("finscope", "a2"), ("finscope", "a6")],
    "4. Poverty and consumption": [("poverty", v) for v in ["cons1", "cons1ae", "sol_jan", "hh_index", "ae", "member", "food",
                                                           "Poverty_line", "Extreme_line", "pov_jan", "epov_jan", "poverty", "quintile"]],
    "5. Food and non-food expenditure": [("poverty", f"exp{i}") for i in range(1, 12)] + [
        ("poverty", "Food_at_school"), ("food", "s8bq0"), ("food", "s8bq4_v2"), ("food", "s8bq8_v2"), ("food", "s8bq10_v2"),
        ("food", "s8bq12_v2"), ("food_away", "s8cq6_v2"), ("nonfood_annual", "s8a1q3"), ("nonfood_monthly", "s8a2q3"),
        ("nonfood_weekly", "s8a3q3"), ("nonfood_weekly", "s08a4_5"), ("other_expenditure", "s9cq2")],
    "6. VUP and social protection": [
        ("vup_direct_support", "s9d1q1"), ("vup_direct_support", "s9d1q2"), ("vup_direct_support", "s9d1q4"),
        ("vup_classic_pw", "s9d2q1"), ("vup_classic_pw", "s9d2q3"), ("vup_classic_pw", "s9d2q11"),
        ("vup_expanded_pw", "s9d3q1"), ("vup_expanded_pw", "s9d3q10"), ("vup_nsds", "s9d4q1"), ("vup_nsds", "s9d4q7"), ("vup_nsds", "s9d4q8"),
        ("vup_financial_services", "s9d5q1"), ("vup_financial_services", "s9d5q4"), ("other_income", "s9eq1"),
        ("household", "s7a2q1"), ("person", "s3q2"), ("person", "s3q3")],
    "7. Financial inclusion": [
        ("savings", "s10cq1"), ("savings", "s10cq4"), ("savings", "s10cq5"), ("savings", "s10cq8"), ("credit", "s10aq1"),
        ("credit", "s10aq3"), ("credit", "s10aq6"), ("person", "s4bq8"), ("person", "s4bq23c"), ("finscope", "k1"),
        ("finscope", "qf3_01"), ("finscope", "qf3_05"), ("finscope", "qf3_06"), ("finscope", "qf4_05"), ("finscope", "f3"),
        ("finscope", "f6"), ("finscope", "g11a"), ("finscope", "h7"), ("finscope", "i4a"), ("finscope", "l1"),
        ("finscope", "e9b"), ("finscope", "e9c")],
    "8. Income and transfers": [("person", "s6bq9a"), ("person", "s6bq9b"), ("other_income", "s9eq4"), ("transfers_in", "s9bq8"),
                                ("transfers_out", "s9aq7"), ("finscope", "n1b"), ("finscope", "n11"), ("finscope", "n12")],
    "8. Employment": [("person", v) for v in ["s6aq2", "s6aq4", "s6aq5", "s6aq6", "s6bq1", "s6bq3", "s6bq4", "s6bq7"]] + [("finscope", "n4")],
    "8. Education": [("person", v) for v in ["s4aq1", "s4aq2", "s4aq3", "s4aq7", "s4bq4"]] + [("finscope", "c4a")],
    "8. Assets": [("durables", "s10bq0"), ("durables", "s10bq2"), ("durables", "s10bq5a"), ("household", "s7aq4"),
                  ("household", "s7aq4a"), ("household", "s7aq4e")],
    "8. Housing and utilities": [("household", v) for v in ["s5aq2", "s5aq3", "s5aq7", "s5dq1", "s5dq2", "s5dq3", "s5cq1",
                                                            "s5cq14", "s5cq16", "s5cq22a", "s5cq25", "s5cq20a"]] + [("finscope", "c7a")],
    "8. Service access": [("services", v) for v in ["s5fq0", "s5fq1", "s5fq3a", "s5fq3b", "s5fq4", "s5fq5"]] + [
        ("finscope", "d1b7"), ("finscope", "d1b14")],
}
key_vars = pd.DataFrame([(area, d, v) for area, items in KEY_VARIABLES.items() for d, v in items],
                        columns=["focus_area", "dataset", "variable"])
key_vars = key_vars.merge(dictionary.drop(columns="focus_area"), on=["dataset", "variable"], how="left", validate="m:1")
assert key_vars["label"].notna().all(), key_vars.loc[key_vars["label"].isna(), ["dataset", "variable"]]
key_vars.to_csv(OUT / "key_variables.csv", index=False)
key_vars[["focus_area", "dataset", "variable", "label", "kind", "pct_missing", "n_unique", "min", "max"]]

## 4. Missingness

High missingness in survey data is usually by design: a question is skipped for people it does not apply to. VUP payment amounts, for example, are blank for households that are not in VUP. The first table counts variables by missingness band. The second shows missingness **among eligible respondents** for key amounts, which is the figure that matters for analysis.

In [ ]:
bands = pd.cut(dictionary["pct_missing"], [-0.01, 0, 5, 50, 95, 99.99, 100],
               labels=["0%", "0-5%", "5-50%", "50-95%", "95-<100%", "100% (empty)"])
missing_by_variable = dictionary.assign(missing_band=bands)[
    ["survey", "dataset", "variable", "label", "focus_area", "role", "kind", "n_rows", "n_missing", "pct_missing", "missing_band"]]
missing_by_variable.to_csv(OUT / "missingness_by_variable.csv", index=False)

missing_by_dataset = (missing_by_variable.pivot_table(index=["survey", "dataset"], columns="missing_band",
                                                      values="variable", aggfunc="count", fill_value=0, observed=False)
                      .join(missing_by_variable.groupby(["survey", "dataset"])["pct_missing"].median().rename("median_pct_missing")))
missing_by_dataset.to_csv(OUT / "missingness_by_dataset.csv")
missing_by_dataset

In [ ]:
def conditional_missing(dataset, amount, condition, value=1):
    f = profiles[dataset].frame
    eligible = f[f[condition] == value]
    return {"dataset": dataset, "amount_variable": amount, "eligible_if": f"{condition} == {value}",
            "eligible_rows": len(eligible), "pct_missing_all_rows": round(100 * f[amount].isna().mean(), 1),
            "pct_missing_among_eligible": round(100 * eligible[amount].isna().mean(), 1) if len(eligible) else np.nan,
            "pct_zero_among_eligible": round(100 * eligible[amount].eq(0).mean(), 1) if len(eligible) else np.nan}

conditional = pd.DataFrame([
    conditional_missing("vup_direct_support", "s9d1q4", "s9d1q2"),
    conditional_missing("vup_classic_pw", "s9d2q11", "s9d2q1"),
    conditional_missing("vup_expanded_pw", "s9d3q10", "s9d3q1"),
    conditional_missing("vup_nsds", "s9d4q8", "s9d4q1"),
    conditional_missing("vup_nsds", "s9d4q8a", "s9d4q1"),
    conditional_missing("vup_financial_services", "s9d5q4", "s9d5q2"),
    conditional_missing("nonfood_annual", "s8a1q3", "s8a1q2"),
    conditional_missing("nonfood_monthly", "s8a2q3", "s8a2q2"),
    conditional_missing("nonfood_weekly", "s8a3q3", "s8a3q2"),
])
conditional.to_csv(OUT / "missingness_conditional.csv", index=False)
conditional

## 5. Quality checks

Each check gets a status. **pass** means nothing to fix. **warn** means fix or handle it before analysis. **info** means a structural fact worth knowing. All results are saved to `quality_checks.csv`.

In [ ]:
checks = []
def check(name, dataset, ok, n_affected=0, detail="", info=False):
    checks.append({"check": name, "dataset": dataset, "status": "info" if info else ("pass" if ok else "warn"),
                   "n_affected": int(n_affected), "detail": detail})

pov = profiles["poverty"].frame.set_index("hhid")
eicv = {n: p for n, p in profiles.items() if p.dataset.survey == "EICV7"}

# Keys and duplicates
for row in inventory.itertuples():
    check("Key is unique", row.dataset, row.key_unique, row.duplicate_key_rows,
          f"key {row.key}; {row.exact_duplicate_rows:,} rows are exact copies of another row")

# Household coverage against the poverty file
for name, p in eicv.items():
    hh = set(p.frame["hhid"])
    missing, extra = len(set(pov.index) - hh), len(hh - set(pov.index))
    check("Covers all poverty-file households", name, missing == 0 and extra == 0, missing + extra,
          f"{missing} poverty-file households absent, {extra} households not in poverty file")

# Weights
w = pov["weight"]
check("Household weight positive and complete", "poverty", w.notna().all() and (w > 0).all(), (w.isna() | (w <= 0)).sum(),
      f"range {w.min():.1f}-{w.max():.1f}; sums to {w.sum():,.0f} households")
check("pop_wt = weight x member", "poverty", np.allclose(pov["pop_wt"], w * pov["member"], rtol=1e-4), 0,
      f"pop_wt sums to {pov['pop_wt'].sum():,.0f} people")
within = pov.groupby("clust")["weight"].nunique()
check("Weight constant within cluster", "poverty", (within == 1).all(), (within > 1).sum(),
      f"{pov['clust'].nunique():,} clusters, about {len(pov) / pov['clust'].nunique():.0f} households each")
nested = pov.groupby("clust")["district"].nunique()
check("Clusters nested in districts", "poverty", (nested == 1).all(), (nested > 1).sum(), "")
check("Stratum identifier location", "all EICV7", True, 0,
      "strata_id is in the module files but not in the poverty or household files; take it from any module", info=True)
check("hhid storage type", "all EICV7", True, 0,
      f"stored as float32; values {int(pov.index.min())}-{int(pov.index.max())} are exact, but cast to int64 before joining",
      info=True)

# Consistency of repeated household variables with the poverty file
for name, p in eicv.items():
    if name == "poverty":
        continue
    f = p.frame.drop_duplicates("hhid").set_index("hhid")
    for var in ["weight", "clust", "province", "district", "ur", "quintile", "poverty", "pov_jan", "epov_jan"]:
        if var not in f:
            continue
        a, b = f[var].astype(float), pov[var].reindex(f.index).astype(float)
        filled = a.notna()
        recoded = var in ("pov_jan", "epov_jan") and a.max() <= 1 < b.max()
        if recoded:
            a = a * 100
        mismatched = filled & ~np.isclose(a, b, rtol=1e-6)
        note = ("coded 0/1 here but 0/100 in the poverty file; " if recoded else "") + (
            f"filled for only {filled.sum():,} of {len(a):,} households (VUP beneficiaries); " if filled.mean() < 0.5 else "")
        check(f"{var} matches poverty file", name, mismatched.sum() == 0 and not recoded, mismatched.sum(),
              note + f"{mismatched.sum()} households differ where filled")

# Aggregate identities in the poverty file
exp_sum = pov[[f"exp{i}" for i in range(1, 12)]].sum(axis=1)
check("cons1 = exp1 + ... + exp11", "poverty", np.allclose(exp_sum, pov["cons1"], atol=10), 0,
      f"max difference {(exp_sum - pov['cons1']).abs().max():.1f} RWF")
food_sum = pov["exp9"] + pov["exp10"] + pov["Food_at_school"]
check("food = exp9 + exp10 + Food_at_school", "poverty", np.allclose(food_sum, pov["food"], atol=10), 0,
      "Food_at_school is also inside exp1 (education), so do not add food and exp1 together", info=False)
check("sol_jan = cons1ae / hh_index", "poverty", np.allclose(pov["cons1ae"] / pov["hh_index"], pov["sol_jan"], rtol=1e-5), 0,
      "hh_index converts each household's consumption to national January 2024 prices")
pov_rebuilt = np.where(pov["sol_jan"] < pov["Poverty_line"], 100, 0)
check("pov_jan = 100 x (sol_jan < Poverty_line)", "poverty", (pov_rebuilt == pov["pov_jan"]).all(),
      (pov_rebuilt != pov["pov_jan"]).sum(), f"poverty line {pov['Poverty_line'].iloc[0]:,.0f} RWF per adult equivalent per year")
person = profiles["person"].frame
n_persons = person.groupby("hhid").size()
check("member = number of person records", "person", (n_persons.reindex(pov.index) == pov["member"]).all(), 0, "")

# Person roster
head_code = next(int(k) for k, v in si.value_labels_for(si.DATASETS_BY_NAME["person"], "s1q2").items() if "head" in str(v).lower())
heads = person[person["s1q2"] == head_code].groupby("hhid").size().reindex(pov.index, fill_value=0)
check("Exactly one household head", "person", (heads == 1).all(), (heads != 1).sum(),
      f"head code {head_code}; {(heads == 0).sum()} households without a head, {(heads > 1).sum()} with more than one")
age = person["s1q3y"]
check("Ages within 0-110", "person", age.between(0, 110).all(), (~age.between(0, 110)).sum(),
      f"range {age.min():.0f}-{age.max():.0f}; {age.isna().sum()} missing")

# Repeated item lists in S9C and S9E
for name, expected in [("other_expenditure", 13), ("other_income", 26)]:
    f = profiles[name].frame
    rows = f.groupby("hhid").size()
    doubled = rows[rows > expected].index
    amount = "s9cq2" if name == "other_expenditure" else "s9eq4"
    item = profiles[name].dataset.item_var
    conflicting = (f[f["hhid"].isin(doubled)].groupby(["hhid", item])[amount].nunique(dropna=True) > 1).sum()
    check("Item list appears once per household", name, len(doubled) == 0, len(doubled),
          f"{len(doubled)} households have {rows.max() // expected}x the {expected}-item list; "
          f"{conflicting} household-items have conflicting amounts")
for name in ["transfers_out", "transfers_in"]:
    p = profiles[name]
    check("No exact duplicate rows", name, p.n_exact_duplicate_rows == 0, p.n_exact_duplicate_rows,
          "deduplicate before summing transfer amounts; up to 99 rows per household")

# VUP files
vup_files = ["vup_direct_support", "vup_classic_pw", "vup_expanded_pw", "vup_nsds", "vup_financial_services"]
for name in vup_files:
    rows = profiles[name].frame.groupby("hhid").size()
    check("One row per household", name, (rows == 1).all(), (rows > 1).sum(),
          f"{(rows > 1).sum()} households have one row per beneficiary member", info=(rows > 1).any())
components = {"vup_direct_support": "s9d1q2", "vup_classic_pw": "s9d2q1", "vup_expanded_pw": "s9d3q1",
              "vup_nsds": "s9d4q1", "vup_financial_services": "s9d5q1"}
in_component = pd.DataFrame({n: profiles[n].frame.groupby("hhid")[c].min().eq(1) for n, c in components.items()})
s9d1q1_yes = pd.DataFrame({n: profiles[n].frame.groupby("hhid")["s9d1q1"].min().eq(1) for n in components})
vup_any = in_component.any(axis=1)
vup_share = 100 * pov["weight"].reindex(vup_any.index)[vup_any].sum() / pov["weight"].sum()
check("s9d1q1 ('beneficiary of any VUP component') is component-specific", "vup_*", True, int(vup_any.sum()),
      f"in every file s9d1q1 = 1 exactly for that file's component (identical: {bool((in_component == s9d1q1_yes).all().all())}); "
      f"{vup_any.sum():,} households ({vup_share:.1f}% weighted) are in at least one component, "
      f"{(in_component.sum(axis=1) >= 2).sum()} in two or more; build vup_any from all five files", info=True)

# Variable-level issues from the dictionary
eicv_dict = dictionary[dictionary["survey"] == "EICV7"]
for survey, d in dictionary.groupby("survey"):
    empty = d[d["pct_missing"] == 100]
    check("No completely empty variables", survey, empty.empty, len(empty), ", ".join((empty["dataset"] + "." + empty["variable"]).head(12)))
    constant = d[(d["n_unique"] == 1) & (d["pct_missing"] < 100)]
    check("Variables with a single value", survey, True, len(constant),
          ", ".join((constant["dataset"] + "." + constant["variable"]).head(12)), info=True)
    has_unlabelled = d["n_unlabelled_values"].fillna(0) > 0
    unlabelled = d[has_unlabelled & (d["n_value_labels"] >= 2) & ~d["role"].isin(["key", "identifier"])]
    check("All codes of categorical variables have labels", survey, unlabelled.empty, len(unlabelled),
          ", ".join((unlabelled["dataset"] + "." + unlabelled["variable"] + " [" + unlabelled["unlabelled_codes"].astype(str) + "]").head(12)))
    special = d[has_unlabelled & (d["n_value_labels"] == 1)]
    check("Numeric variables that label one special value only", survey, True, len(special),
          ", ".join((special["dataset"] + "." + special["variable"] + " [" + special["value_labels"].astype(str).str[:30] + "]").head(8)),
          info=True)
    dk = d[d["n_dk_refused"].fillna(0) > 0]
    check("Don't know / refused codes present", survey, True, len(dk),
          f"{len(dk)} variables use a labelled don't know or refused code; treat as missing in analysis", info=True)
    negative = d[(d["n_negative"].fillna(0) > 0) & (d["n_value_labels"] == 0)]
    check("No negative values in unlabelled numeric variables", survey, negative.empty, len(negative),
          ", ".join((negative["dataset"] + "." + negative["variable"]).head(10)))
    sentinel = d[d["n_possible_sentinel"].fillna(0) > 0]
    check("Possible sentinel values (9999, -99, ...) in unlabelled variables", survey, True, len(sentinel),
          ", ".join((sentinel["dataset"] + "." + sentinel["variable"]).head(10)), info=True)

# Text encoding
latin1 = [n for n, p in profiles.items() if p.latin1_fallback]
check("Labels are valid UTF-8", ", ".join(latin1) or "all", not latin1, len(latin1),
      "some value or variable labels needed latin-1 decoding; check accented or curly-quote characters in labels")

# FinScope
fs = profiles["finscope"].frame
check("Key is unique", "finscope", fs["key"].is_unique, fs["key"].duplicated().sum(), "")
check("Geography codes match EICV7", "finscope",
      set(fs["a2"]) <= set(pov["district"]) and set(fs["a6"]) <= set(pov["ur"]) and set(fs["a1"]) <= set(pov["province"]), 0,
      "same province, district (11-57) and urban/rural codes as EICV7")
cells = fs.groupby(["a2", "a6"]).size()
check("Respondents per district x urban/rural cell", "finscope", True, len(cells),
      f"{len(cells)} cells, {cells.min()}-{cells.max()} respondents (median {cells.median():.0f})", info=True)
check("Weighted population totals", "finscope", True, 0,
      f"hh_wt sums to {fs['hh_wt'].sum():,.0f} households (EICV7: {pov['weight'].sum():,.0f}); "
      f"pop_wt sums to {fs['pop_wt'].sum():,.0f} adults aged 16+", info=True)
check("No interview date in either survey", "EICV7 and FinScope", True, 0,
      "No interview date or month variable in any file; EICV7 values are in January 2024 prices via hh_index", info=True)

quality = pd.DataFrame(checks)
quality.to_csv(OUT / "quality_checks.csv", index=False)
print(quality["status"].value_counts().to_string())
quality[quality["status"] != "pass"]

In [ ]:
quality[quality["status"] == "pass"][["check", "dataset", "detail"]]

## 6. Consumption structure and the CPI basket

The CPI weights come from EICV4 (2013/14). The table compares them with EICV7 budget shares, computed from NISR's consumption aggregates (each component's weighted total as a share of total consumption). It is an approximate comparison. EICV7 components do not line up exactly with CPI divisions: `exp9` includes alcohol and own-produced food, and `Food_at_school` sits inside education.

In [ ]:
pf = profiles["poverty"].frame
components = {"exp9": "Food consumed at home (exp9)", "exp10": "Food away from home (exp10)", "exp1": "Education (exp1)",
              "exp2": "Imputed rent (exp2)", "exp3": "Actual rent (exp3)", "exp4": "Water (exp4)", "exp5": "Electricity (exp5)",
              "exp6": "Annual non-food items (exp6)", "exp7": "Monthly non-food items (exp7)",
              "exp8": "Frequent non-food items (exp8)", "exp11": "Use value of durables (exp11)"}
areas = {"urban": pf[pf["ur"] == 1], "rural": pf[pf["ur"] == 2], "national": pf}
budget = pd.DataFrame({
    area: {label: 100 * (g[c] * g["weight"]).sum() / (g["cons1"] * g["weight"]).sum() for c, label in components.items()}
    for area, g in areas.items()
})
budget.loc["Total"] = budget.sum()
budget = budget.round(1)

cpi_path = ROOT / "data" / "processed" / "cpi_features.parquet"
if cpi_path.exists():
    cpi = pd.read_parquet(cpi_path, columns=["area", "category", "coicop", "source_sheet", "weight"])
    cpi = cpi[cpi["source_sheet"] != "Other_Indices"].drop_duplicates(["area", "category"])
    w = cpi.pivot(index="category", columns="area", values="weight") / 100
    print("CPI weights (EICV4 basket, %):")
    display(w.loc[["food_non_alcoholic_beverages", "alcohol_tobacco", "restaurants_hotels", "housing_utilities", "education"],
                  ["urban", "rural", "national"]].round(1))
budget

## 7. Expenditure items and the CPI mapping

Each EICV7 expenditure item is mapped to the most detailed CPI series NISR publishes (see `src/data/eicv7_cpi_mapping.py`). The mapping first assigns each item a COICOP 1999 group, the classification the CPI uses. EICV7 itself uses some COICOP 2018 codes. The raw statistics below show how much each item matters: the number of households reporting a purchase, and each item's share of weighted purchase value within its module. These figures are **raw**: not annualised, not deduplicated and not cleaned. Use them only to decide where the mapping matters most.

In [ ]:
value_labels = {m: si.value_labels_for(si.DATASETS_BY_NAME[m], var) for m, (var, *_) in cpimap.ITEM_VARS.items()}
mapping = cpimap.build_mapping(value_labels)

AMOUNTS = {"food": [f"s8bq4_v{v}" for v in range(2, 6)], "nonfood_annual": ["s8a1q3"], "nonfood_monthly": ["s8a2q3"],
           "nonfood_weekly": ["s8a3q3"], "other_expenditure": ["s9cq2"], "durables": ["s10bq5a"]}
raw = []
for module, cols in AMOUNTS.items():
    f = profiles[module].frame
    item = profiles[module].dataset.item_var
    amount = f[cols].clip(lower=0).sum(axis=1, min_count=1)
    reported = f.loc[amount > 0, [item, "hhid"]]
    stats = pd.DataFrame({
        "n_households_reporting": reported.groupby(item)["hhid"].nunique(),
        "weighted_value": (amount * f["weight"]).groupby(f[item]).sum(),
    })
    stats["raw_weighted_share_of_module_pct"] = (100 * stats["weighted_value"] / stats["weighted_value"].sum()).round(2)
    raw.append(stats.drop(columns="weighted_value").reset_index(names="item_code").assign(dataset=module))
raw = pd.concat(raw)
raw["item_code"] = raw["item_code"].astype("Int64")
mapping = mapping.merge(raw, on=["dataset", "item_code"], how="left")
mapping.to_csv(OUT / "expenditure_cpi_mapping.csv", index=False)
print(f"{len(mapping)} items or variables mapped")
mapping.pivot_table(index="cpi_category", columns="cpi_match_level", values="item_label", aggfunc="count", fill_value=0)

In [ ]:
# Where the food mapping matters most: share of raw purchase value by CPI series
food_map = mapping[mapping["dataset"] == "food"]
(food_map.groupby(["cpi_category", "cpi_match_level"])["raw_weighted_share_of_module_pct"].sum()
 .sort_values(ascending=False).round(1).to_frame("share_of_food_purchases_pct"))

In [ ]:
# The ten largest items in each item module
(mapping.dropna(subset=["raw_weighted_share_of_module_pct"]).sort_values("raw_weighted_share_of_module_pct", ascending=False)
 .groupby("module").head(10)
 [["module", "item_code", "item_label", "coicop_1999_group", "cpi_category", "cpi_match_level", "alt_cpi_category",
   "n_households_reporting", "raw_weighted_share_of_module_pct"]]
 .sort_values(["module", "raw_weighted_share_of_module_pct"], ascending=[True, False]))

## 8. Recommended household-level analysis table

The proposal is **one row per EICV7 household** (15,054 rows), keyed by `hhid`, starting from the poverty file. Person-level and item-level modules are aggregated to the household. FinScope cannot be matched to individual households because it is a different sample. It is joined as **district × urban/rural indicators** instead. CPI series join on urban/rural status. EICV7 has no interview month, so the CPI join is anchored at NISR's January 2024 price base. The full column list is in `household_table_design.csv`, and the reasoning is in `reports/eicv7_finscope_inventory.md`.

In [ ]:
D = []
def col(block, name, source, variables, rule, notes=""):
    D.append({"block": block, "column": name, "source_dataset": source, "source_variables": variables, "rule": rule, "notes": notes})

col("Keys and design", "hhid", "poverty", "hhid", "cast float32 to int64", "Primary key; every module joins on it.")
col("Keys and design", "clust", "poverty", "clust", "as is", "Primary sampling unit for standard errors.")
col("Keys and design", "strata_id", "any S8-S10 module", "strata_id", "first value per hhid", "Not in the poverty or household files.")
col("Keys and design", "province, district, ur", "poverty", "province, district, ur", "as is", "Same codes as FinScope a1, a2, a6.")
col("Keys and design", "weight", "poverty", "weight", "as is", "Household weight; sums to about 3.29 million households.")
col("Keys and design", "pop_wt", "poverty", "pop_wt", "as is", "= weight x member; use for person-level rates such as poverty.")
col("Welfare", "member, ae", "poverty", "member, ae", "as is", "Household size and adult equivalents.")
col("Welfare", "cons_nominal, cons_nominal_ae", "poverty", "cons1, cons1ae", "rename", "Annual consumption in survey-period prices (RWF).")
col("Welfare", "hh_price_index", "poverty", "hh_index", "rename", "Household price deflator to national January 2024 prices.")
col("Welfare", "cons_real_ae_jan2024", "poverty", "sol_jan", "rename", "= cons1ae / hh_index. NISR's welfare measure.")
col("Welfare", "cons_real_jan2024", "poverty", "cons1, hh_index", "cons1 / hh_index", "")
col("Welfare", "exp_education ... exp_durables_use", "poverty", "exp1-exp11, Food_at_school", "rename", "Components of cons1; nominal.")
col("Welfare", "food_share", "poverty", "food, cons1", "food / cons1", "food = exp9 + exp10 + Food_at_school.")
col("Welfare", "poor, extreme_poor", "poverty", "pov_jan, epov_jan", "divide by 100 -> 0/1", "Module files code these 0/1, the poverty file 0/100.")
col("Welfare", "poverty_status, quintile", "poverty", "poverty, quintile", "as is", "Value labels in the data dictionary.")
col("CPI linkage", "cpi_area", "derived", "ur", "1 -> urban, 2 -> rural", "Selects the CPI series.")
col("CPI linkage", "budget_share_<cpi_category>", "S8A1-S8A3, S8B, S8C, S10B, poverty", "item amounts + expenditure_cpi_mapping.csv",
    "annualise each module's recall, sum by cpi_category, divide by total", "Household-specific CPI weights. Use exp1-exp11 to scale modules to NISR totals.")
col("CPI linkage", "cpi_change_jan2024_sep2026_<category>", "cpi_features.parquet", "index_value",
    "index(2026-09) / index(2024-01) - 1, by cpi_area", "January 2024 is the price base of sol_jan.")
col("CPI linkage", "hh_inflation_jan2024_sep2026", "derived", "budget shares x category changes", "sum over categories",
    "Inflation exposure of each household's own basket.")
col("CPI linkage", "cons_real_ae_sep2026", "derived", "sol_jan, headline CPI change", "sol_jan x (1 + headline change for cpi_area)",
    "Optional: express welfare in current prices.")
col("Demographics", "head_sex, head_age, head_marital_status", "person", "s1q1, s1q3y, s1q4 where s1q2 = head", "head's values", "One head per household (checked).")
col("Demographics", "n_children_0_14, n_adults_15_64, n_elderly_65plus, dependency_ratio", "person", "s1q3y", "counts", "")
col("Demographics", "n_members_with_disability", "person", "s3q7-s3q12", "count of members with any 'a lot of difficulty' or 'cannot do'", "Washington Group short set.")
col("Education", "head_education_level, head_literate", "person", "s4aq2, s4aq3, s4bq4", "head's values", "")
col("Education", "max_education_any_member, n_children_in_school", "person", "s4aq2, s4aq7, s1q3y", "max / count", "")
col("Employment", "head_employed, head_sector, head_status", "person", "s6aq2-s6aq9, s6bq4, s6bq7", "head's values", "ISIC level 1 in s6bq4.")
col("Employment", "n_employed, n_wage_workers, any_nonfarm_business", "person", "s6aq2-s6aq7", "counts / any", "")
col("Income", "wage_earnings_annual", "person", "s6bq9a, s6bq9b", "convert amount by unit to annual, sum over members", "Needs unit harmonisation; main job only.")
col("Income", "income_<source group>_12m", "other_income", "s9eq1, s9eq4", "deduplicate, then sum by group",
    "Groups: pensions (1, 12, 13), public transfers (3-5, 7, 8, 10, 22), farm (24-26), property and financial (17, 18, 20, 21), other.")
col("Income", "remittances_received_12m", "transfers_in", "s9bq8", "deduplicate, then sum", "")
col("Income", "transfers_sent_12m", "transfers_out", "s9aq7, s9aq10, s9aq12", "deduplicate, then sum", "")
col("Social protection", "vup_any", "vup_* files", "s9d1q2, s9d2q1, s9d3q1, s9d4q1, s9d5q1", "1 if any component flag is yes",
    "Do not use s9d1q1: despite its label it only flags the component of the file it is in.")
col("Social protection", "vup_direct_support, vup_ds_amount_12m", "vup_direct_support", "s9d1q2, s9d1q4", "any / sum over beneficiary rows", "")
col("Social protection", "vup_classic_pw, vup_cpw_months, vup_cpw_amount_12m", "vup_classic_pw", "s9d2q1, s9d2q3, s9d2q11", "as is", "")
col("Social protection", "vup_expanded_pw, vup_epw_amount_12m", "vup_expanded_pw", "s9d3q1, s9d3q10", "any / sum", "")
col("Social protection", "nsds, nsds_quarters_paid, nsds_amount", "vup_nsds", "s9d4q1, s9d4q7, s9d4q8", "any / as is",
    "s9d4q8 is labelled 'first quarter' but is the only filled amount; s9d4q8a-d are almost empty. Confirm the definition in the questionnaire.")
col("Social protection", "vup_fs_applied, vup_fs_loan_amount", "vup_financial_services", "s9d5q1, s9d5q2, s9d5q4", "as is", "")
col("Social protection", "girinka_cow", "household", "s7a2q1", "1 if yes", "One Cow per Poor Family.")
col("Social protection", "share_members_health_insured", "person", "s3q2", "share of members", "")
col("Financial inclusion (EICV7)", "any_account, any_bank_account, any_sacco_account, any_mobile_money_account", "savings",
    "s10cq4, s10cq5", "any member", "Institution codes: 1-3 bank/MFI, 4 SACCO, 5-6 mobile money.")
col("Financial inclusion (EICV7)", "any_tontine", "savings", "s10cq8", "any member", "")
col("Financial inclusion (EICV7)", "any_loan_outstanding, any_formal_loan, n_loans", "credit", "s10aq3, s10aq6", "any / count", "Formal: bank, MFI, SACCO, cooperative, VUP.")
col("Financial inclusion (EICV7)", "any_member_mobile_money_use, any_member_phone", "person", "s4bq23c, s4bq8", "any member", "")
col("Financial inclusion (FinScope)", "fs_<indicator>_district_ur", "finscope", "qf3_*, qf4_*, k1, f3, f6, l1, g11a, i4a, h3",
    "pop_wt-weighted share of adults, by a2 x a6", "Area-level context only; FinScope is a different sample. Cells have 96-607 respondents.")
col("Housing and utilities", "dwelling_type, tenure, n_rooms", "household", "s5aq2, s5aq7, s5aq8, s5aq3", "as is", "")
col("Housing and utilities", "wall, roof, floor material", "household", "s5dq1, s5dq2, s5dq3", "as is", "")
col("Housing and utilities", "drinking_water_source, grid_connected, lighting_source, cooking_fuel, toilet_type, internet_at_home",
    "household", "s5cq3, s5cq14, s5cq16, s5cq22a, s5cq25, s5cq20a", "as is", "")
col("Assets", "owns_<durable>, n_<durable>", "durables", "s10bq0, s10bq2", "pivot wide by item", "Radio, TV, phones, computer, fridge, bicycle, motorcycle, car.")
col("Assets", "durables_current_value", "durables", "s10bq5a-c", "sum", "")
col("Assets", "n_cattle ... n_poultry, tropical_livestock_units", "household", "s7aq4a-f", "as is; TLU weights", "")
col("Service access", "minutes_to_<service>", "services", "s5fq0, s5fq3a, s5fq3b", "hours x 60 + minutes, pivot wide by service",
    "Water, food market, public transport, primary school, health centre, sector office.")
col("Shocks", "any_shock_12m, main_shock", "household", "s5eq1, s5eq2a", "as is", "")

design = pd.DataFrame(D)
design.to_csv(OUT / "household_table_design.csv", index=False)
design

In [ ]:
sorted(p.name for p in OUT.glob("*.csv"))